# 03 — Evaluar RAG con RAGAS

Evalúa la calidad del pipeline RAG usando el gold standard.  
**Entrada:** `data/08_reporting/seleccion_30.csv`  
**Salida:** `data/08_reporting/results_<experiment_id>_<timestamp>.csv`

---

## Fases y métricas

| Fase | Estado | Métricas |
|------|--------|----------|
| **Retrieval** | ✅ activo | Context Recall, Context Precision, Context Entities Recall |
| **Generation** | 🔜 pendiente | Faithfulness, Response Relevancy, Noise Sensitivity |
| **NVIDIA** | 🔜 pendiente | Answer Correctness, Answer Coherence, Answer Fluency |

## Uso

1. Ajusta la sección **CONFIGURACIÓN** (experiment ID, top_k, modelos).
2. Ejecuta todas las celdas hasta la sección de **Generación** (inclusive solo si `EVAL_MODE != "retrieval"`).
3. Los resultados se guardan automáticamente en `08_reporting/`.

In [ ]:
%pip install ragas langchain-ollama qdrant-client pandas tqdm requests --quiet

In [ ]:

# Workaround: ragas 0.4.x importa langchain_community.chat_models.vertexai
# que fue eliminado en langchain-community 0.3+. Como no usamos VertexAI,
# basta con registrar un módulo stub para satisfacer el import.
import types, sys
if "langchain_community.chat_models.vertexai" not in sys.modules:
    _stub = types.ModuleType("langchain_community.chat_models.vertexai")
    _stub.ChatVertexAI = None
    sys.modules["langchain_community.chat_models.vertexai"] = _stub
if "langchain_community.llms.vertexai" not in sys.modules:
    _stub2 = types.ModuleType("langchain_community.llms.vertexai")
    _stub2.VertexAI = None
    sys.modules["langchain_community.llms.vertexai"] = _stub2

import ast
import json
import getpass
import os
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from tqdm.auto import tqdm
from qdrant_client import QdrantClient

from langchain_ollama import ChatOllama, OllamaEmbeddings
from ragas import EvaluationDataset, SingleTurnSample, evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    ContextEntityRecall,
)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURACIÓN — ajusta aquí antes de ejecutar
# ══════════════════════════════════════════════════════════════════════════════

# ── Estrategia de chunking ─────────────────────────────────────────────────────
CHUNK_STRATEGY = "fixed_256"

# ── Datos ──────────────────────────────────────────────────────────────────────
GOLD_PATH    = "/home/coder/ia-testing/rageval/data/08_reporting/seleccion_30.csv"
RESULTS_DIR_STR = "/home/coder/ia-testing/rageval/data/08_reporting"

# ── Retrieval ──────────────────────────────────────────────────────────────────
QDRANT_URL = "https://endorqdrant.altia.es:443"
TOP_K      = 5

# ── Modelos ────────────────────────────────────────────────────────────────────
OLLAMA_URL      = "http://localhost:11434"
EMBED_MODEL     = "nomic-embed-text"
JUDGE_LLM_MODEL = "qwen3.5:4b-q4_K_M"

# ── Subconjunto de evaluación ──────────────────────────────────────────────────
N_SAMPLES = None

# ── Modo de evaluación ─────────────────────────────────────────────────────────
EVAL_MODE = "full"

# ── Columnas de metadata del gold que se propagan al CSV de resultados ─────────
METADATA_COLS = ["query_type", "query_style", "category", "doc_id", "section_id"]
# ══════════════════════════════════════════════════════════════════════════════
# Nota: EXPERIMENT_ID, EXPERIMENT_NOTES y COLLECTION_NAME se derivan
# automáticamente de CHUNK_STRATEGY + EMBED_MODEL en la celda de derivación
# (misma fuente de verdad que usa nb02 al nombrar la colección).

In [ ]:
# Derivación — corre DESPUÉS del override de papermill
import re as _re

def _model_slug(model: str) -> str:
    """avr/sfr-embedding-mistral -> sfrembeddingmistral"""
    name = model.split("/")[-1]                    # quita prefijo usuario/org
    name = _re.sub(r"[^a-zA-Z0-9]+", "", name)      # quita guiones, puntos...
    return name.lower()

RESULTS_DIR      = Path(RESULTS_DIR_STR)
COLLECTION_NAME  = f"altia_rag_{_model_slug(EMBED_MODEL)}_{CHUNK_STRATEGY}"
EXPERIMENT_ID    = f"{CHUNK_STRATEGY}_{_model_slug(EMBED_MODEL)}_v0"
EXPERIMENT_NOTES = f"{CHUNK_STRATEGY} chunks + {EMBED_MODEL} + cosine, top_k={TOP_K}"

print(f"Estrategia  : {CHUNK_STRATEGY}")
print(f"Experimento : {EXPERIMENT_ID}")
print(f"Notas       : {EXPERIMENT_NOTES}")
print(f"Modo        : {EVAL_MODE}")
print(f"Colección   : {COLLECTION_NAME}  top_k={TOP_K}")
print(f"Embedding   : {EMBED_MODEL}")
print(f"LLM juez    : {JUDGE_LLM_MODEL}")
print(f"N_SAMPLES   : {N_SAMPLES if N_SAMPLES is not None else 'todas'}")

## 1. Cargar gold standard

In [4]:

gold_df = pd.read_csv(GOLD_PATH)
gold_df["reference_contexts"] = gold_df["reference_contexts"].apply(ast.literal_eval)

n_total = len(gold_df)
if N_SAMPLES is not None:
    gold_df = gold_df.head(N_SAMPLES).reset_index(drop=True)

print(f"Gold standard: {len(gold_df)} preguntas  (total disponibles: {n_total})")
print(f"Columnas     : {list(gold_df.columns)}")
print()
print("Distribución por query_type:")
print(gold_df["query_type"].value_counts().to_string())
print()
print("Distribución por category:")
print(gold_df["category"].value_counts().to_string())

gold_df[["user_input", "reference", "doc_id", "query_type", "category"]].head(5)


Gold standard: 30 preguntas  (total disponibles: 30)
Columnas     : ['user_input', 'reference', 'reference_contexts', 'query_type', 'query_style', 'category', 'section_id', 'section_title', 'doc_id']

Distribución por query_type:
query_type
single_hop_specific    16
multi_hop_abstract      8
multi_hop_specific      6

Distribución por category:
category
otros           5
corporativo     4
calidad         3
ambiental       3
incidencias     3
financiero      3
rrhh            3
proyectos       3
estrategicos    2
seguridad       1


,user_input,reference,doc_id,query_type,category
0,¿Quién es responsable de ejecutar el inventari...,El Coordinador Ambiental de la oficina será el...,P_19_03,single_hop_specific,ambiental
1,Quien nombra a los Coordinadores de Calidad y ...,"Los Coordinadores de Calidad, Coordinador Ambi...",IT_01_03,single_hop_specific,calidad
2,¿Cuál es el coste de la aplicación DAVdroid?,La aplicación DAVdroid tiene un coste.,IT_24_06,single_hop_specific,corporativo
3,¿Quién es el único rol que puede asumir la fun...,Este rol solo podrá ser asignado a Gestores de...,IT_24_05,single_hop_specific,corporativo
4,¿En qué casos puede un Consejero revelar infor...,Un Consejero puede revelar información confide...,IT_22_01,single_hop_specific,estrategicos


## 2. Conectar a Qdrant y preparar embedding

In [5]:
if EVAL_MODE == "generation":
    print("[SKIP] EVAL_MODE='generation' — conexión a Qdrant omitida.")
else:
    QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY") or getpass.getpass("QDRANT_API_KEY: ")
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, check_compatibility=False)

    info = client.get_collection(COLLECTION_NAME)
    print(f"Colección  : {COLLECTION_NAME}")
    print(f"Vectores   : {info.points_count}")
    print(f"Dimensión  : {info.config.params.vectors.size}")
    print(f"Distancia  : {info.config.params.vectors.distance}")


[SKIP] EVAL_MODE='generation' — conexión a Qdrant omitida.


In [6]:
if EVAL_MODE == "generation":
    print("[SKIP] EVAL_MODE='generation' — embed_query no necesaria.")
else:
    def embed_query(text: str) -> list[float]:
        resp = requests.post(
            f"{OLLAMA_URL}/api/embed",
            json={"model": EMBED_MODEL, "input": text},
            timeout=120,
        )
        resp.raise_for_status()
        return resp.json()["embeddings"][0]

    test_vec = embed_query("prueba de conexión")
    print(f"Embedding OK — dimensión: {len(test_vec)}")


[SKIP] EVAL_MODE='generation' — embed_query no necesaria.


## 3. Retrieval — recuperar contextos para cada pregunta

In [ ]:
if EVAL_MODE == "generation":
    print("[SKIP] EVAL_MODE='generation' — loop de retrieval omitido. Carga desde cache.")
else:
    retrieval_rows = []

    for _, row in tqdm(gold_df.iterrows(), total=len(gold_df), desc="Recuperando contextos"):
        query_vec = embed_query(row["user_input"])
        results = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vec,
            limit=TOP_K,
        ).points

        retrieval_rows.append({
            "user_input":          row["user_input"],
            "reference":           row["reference"],
            "reference_contexts":  row["reference_contexts"],
            "retrieved_contexts":  [
                r.payload.get("parent_text") or r.payload.get("page_content", "")
                for r in results
            ],
            "retrieved_chunk_ids": [r.payload.get("chunk_id", str(r.id)) for r in results],
            "retrieved_scores":    [round(r.score, 4) for r in results],
            **{col: row.get(col, "") for col in METADATA_COLS},
        })

    print(f"\nRetrieval completado: {len(retrieval_rows)} preguntas")
    print(f"\nEjemplo — '{retrieval_rows[0]['user_input'][:60]}...'")
    print(f"  Top-1 score : {retrieval_rows[0]['retrieved_scores'][0]:.4f}")
    print(f"  Top-1 chunk : {retrieval_rows[0]['retrieved_chunk_ids'][0]}")

In [8]:
if EVAL_MODE == "generation":
    print("[SKIP] EVAL_MODE='generation' — guardar cache retrieval omitido.")
else:
    CACHE_PATH = RESULTS_DIR / f"retrieval_{EXPERIMENT_ID}.json"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(retrieval_rows, f, ensure_ascii=False, indent=2, default=str)
    print(f"Cache guardado: {CACHE_PATH}  ({len(retrieval_rows)} filas)")


[SKIP] EVAL_MODE='generation' — guardar cache retrieval omitido.


In [9]:

# ── Cargar retrieval desde disco (ejecuta solo esta celda para saltar el loop) ─
CACHE_PATH = RESULTS_DIR / f"retrieval_{EXPERIMENT_ID}.json"
with open(CACHE_PATH, encoding="utf-8") as f:
    retrieval_rows = json.load(f)
print(f"Cache cargado: {CACHE_PATH}  ({len(retrieval_rows)} filas)")


Cache cargado: ../../rageval/data/08_reporting/retrieval_baseline_v0.json  (30 filas)


## 4. Construir EvaluationDataset

In [10]:
if EVAL_MODE == "generation":
    print("[SKIP] EVAL_MODE='generation' — dataset de retrieval omitido.")
else:
    samples = [
        SingleTurnSample(
            user_input=row["user_input"],
            retrieved_contexts=row["retrieved_contexts"],
            reference_contexts=row["reference_contexts"],
            reference=row["reference"],
        )
        for row in retrieval_rows
    ]
    eval_dataset = EvaluationDataset(samples=samples)
    print(f"EvaluationDataset: {len(eval_dataset)} muestras")


[SKIP] EVAL_MODE='generation' — dataset de retrieval omitido.


In [11]:
if EVAL_MODE == "generation":
    print("[SKIP] EVAL_MODE='generation'")
else:
    print(json.dumps(retrieval_rows[0], indent=2, ensure_ascii=False, default=str))


[SKIP] EVAL_MODE='generation'


## 5. Configurar LLM juez (RAGAS)

In [12]:

# reasoning=False: desactiva el thinking de qwen3.5 (parámetro correcto en
#                  langchain-ollama>=1.1.0; think=False era ignorado silenciosamente)
# num_ctx=4096  : CRÍTICO — qwen3.5 tiene 256K ctx por defecto y Ollama reserva
#                  todo el KV cache (18 GB VRAM). Con 4096 baja a ~300 MB y la
#                  inferencia es x10 más rápida. Los prompts de ragas no superan 4K tokens.
# num_predict=4096: suficiente para los JSON estructurados de ragas
judge_llm = LangchainLLMWrapper(
    ChatOllama(
        model=JUDGE_LLM_MODEL,
        base_url=OLLAMA_URL,
        reasoning=False,
        num_ctx=4096,
        num_predict=4096,
    )
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_URL)
)

print(f"LLM juez    : {JUDGE_LLM_MODEL}")
print(f"Embeddings  : {EMBED_MODEL} vía OllamaEmbeddings")
print(f"reasoning=False  |  num_ctx=4096  |  num_predict=4096")

run_cfg = RunConfig(max_workers=1, timeout=900)
print(f"RunConfig  : max_workers={run_cfg.max_workers}  timeout={run_cfg.timeout}s")

LLM juez    : qwen3.5:4b-q4_K_M
Embeddings  : nomic-embed-text vía OllamaEmbeddings
reasoning=False  |  num_ctx=4096  |  num_predict=4096
RunConfig  : max_workers=1  timeout=900s


/tmp/ipykernel_2140688/1435349043.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(
/tmp/ipykernel_2140688/1435349043.py:17: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


In [13]:
import json as _json
from ragas.prompt.pydantic_prompt import RagasOutputParser
from ragas.prompt.utils import extract_json
from langchain_core.output_parsers import PydanticOutputParser

if not getattr(RagasOutputParser, '_patched', False):
    _orig_parse = RagasOutputParser.parse_output_string

    async def _patched_parse(self, output_string, prompt_value, llm, callbacks, retries_left=1):
        try:
            jsonstr = extract_json(output_string or "")
            if jsonstr and jsonstr.strip().startswith("["):
                data = _json.loads(jsonstr)
                for field_name in self.pydantic_object.model_fields:
                    try:
                        wrapped = _json.dumps({field_name: data})
                        return PydanticOutputParser(pydantic_object=self.pydantic_object).parse(wrapped)
                    except Exception:
                        continue
        except Exception:
            pass
        return await _orig_parse(self, output_string, prompt_value, llm, callbacks, retries_left)

    RagasOutputParser.parse_output_string = _patched_parse
    RagasOutputParser._patched = True
    print("RAGAS parser patched — arrays JSON auto-envueltos")
else:
    print("RAGAS parser ya estaba parchado — skip")


RAGAS parser patched — arrays JSON auto-envueltos


## 6. Evaluar — métricas de retrieval

| Métrica | Qué mide |
|---------|----------|
| **Context Recall** | ¿Se recupera toda la información necesaria para responder? |
| **Context Precision** | ¿Proporción del contexto recuperado que es relevante? |
| **Context Entities Recall** | ¿Se recuperan las entidades clave del ground truth? |

In [14]:
if EVAL_MODE == "generation":
    print("[SKIP] EVAL_MODE='generation' — evaluación de retrieval omitida.")
else:
    retrieval_metrics = [
        LLMContextRecall(llm=judge_llm),
        LLMContextPrecisionWithReference(llm=judge_llm),
        ContextEntityRecall(llm=judge_llm),
    ]

    metric_names = [m.name for m in retrieval_metrics]
    print(f"Métricas   : {metric_names}")
    print(f"Muestras   : {len(eval_dataset)}")
    print(f"LLM juez   : {JUDGE_LLM_MODEL}")
    print(f"Timeout    : {run_cfg.timeout}s")
    print()

    retrieval_result = evaluate(
        dataset=eval_dataset,
        metrics=retrieval_metrics,
        run_config=run_cfg,
    )

    print("\n=== RESULTADOS RETRIEVAL ===")
    print(retrieval_result)


[SKIP] EVAL_MODE='generation' — evaluación de retrieval omitida.


## 7. Resultados

In [15]:
if EVAL_MODE == "generation":
    print("[SKIP] EVAL_MODE='generation' — tablas de retrieval omitidas.")
else:
    scores_df = retrieval_result.to_pandas()

    meta_df = pd.DataFrame(retrieval_rows)[METADATA_COLS + ["retrieved_chunk_ids", "retrieved_scores"]]
    meta_df["retrieved_chunk_ids"] = meta_df["retrieved_chunk_ids"].apply(json.dumps)
    meta_df["retrieved_scores"]    = meta_df["retrieved_scores"].apply(json.dumps)
    scores_df = pd.concat([scores_df, meta_df.reset_index(drop=True)], axis=1)

    print("── Scores por pregunta ──────────────────────────────────")
    display(scores_df[["user_input"] + metric_names + ["category", "doc_id"]].round(4))

    print()
    print("── Agregados ────────────────────────────────────────────")
    display(scores_df[metric_names].agg(["mean", "std", "min", "max"]).round(4))

    print()
    print("── Por categoría ────────────────────────────────────────")
    display(scores_df.groupby("category")[metric_names].mean().round(4))

    print()
    print("── Por query_type ───────────────────────────────────────")
    display(scores_df.groupby("query_type")[metric_names].mean().round(4))


[SKIP] EVAL_MODE='generation' — tablas de retrieval omitidas.


## 8. Guardar resultados

In [16]:
if EVAL_MODE == "generation":
    print("[SKIP] EVAL_MODE='generation' — guardar resultados retrieval omitido.")
else:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    out_path  = RESULTS_DIR / f"results_{EXPERIMENT_ID}_{timestamp}.csv"

    scores_df.insert(0, "experiment_id",    EXPERIMENT_ID)
    scores_df.insert(1, "experiment_notes", EXPERIMENT_NOTES)
    scores_df.insert(2, "eval_timestamp",   timestamp)
    scores_df.insert(3, "embed_model",      EMBED_MODEL)
    scores_df.insert(4, "judge_llm",        JUDGE_LLM_MODEL)
    scores_df.insert(5, "collection",       COLLECTION_NAME)
    scores_df.insert(6, "top_k",            TOP_K)

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    scores_df.to_csv(out_path, index=False)

    print(f"Guardado : {out_path}")
    print(f"Filas    : {len(scores_df)}")
    print(f"Columnas : {len(scores_df.columns)}")
    print()

    summary = {"experiment_id": EXPERIMENT_ID, "timestamp": timestamp,
               "embed_model": EMBED_MODEL, "top_k": TOP_K, "n_samples": len(scores_df)}
    for m in metric_names:
        summary[m] = round(scores_df[m].mean(), 4)

    summary_path = RESULTS_DIR / "scores_summary.csv"
    summary_df   = pd.DataFrame([summary])
    if summary_path.exists():
        existing = pd.read_csv(summary_path)
        existing = existing[existing["experiment_id"] != EXPERIMENT_ID]
        summary_df = pd.concat([existing, summary_df], ignore_index=True)
    summary_df.to_csv(summary_path, index=False)

    print(f"Resumen acumulado: {summary_path}")
    display(summary_df)


[SKIP] EVAL_MODE='generation' — guardar resultados retrieval omitido.


---

# Generación

Activa con `EVAL_MODE = "generation"` o `"full"` en la celda de configuración.  
En modo `"generation"` las secciones 2–8 se saltan automáticamente y se carga el cache de retrieval.


## 9. Configurar generador

| Parámetro | Valor |
|-----------|-------|
| `LLM_GENERATOR_MODEL` | Modelo generador (puede ser distinto del juez) |
| `GEN_NUM_CTX` | Ventana de contexto — 8192 cubre 5 chunks + prompt |
| `GEN_NUM_PREDICT` | Máx. tokens de salida — 512 suficiente para RAG |
| `reasoning=False` | Desactiva el bloque `<think>` de qwen3 |


In [ ]:
if EVAL_MODE not in ("generation", "full"):
    print("[SKIP] EVAL_MODE='retrieval' — sección de generación desactivada.")
else:
    LLM_GENERATOR_MODEL = "qwen3.5:4b-q4_K_M"
    SYSTEM_PROMPT = (
        "Eres un asistente experto en los procesos internos de la empresa. "
        "Responde únicamente basándote en el contexto proporcionado. "
        "Si la información no está en el contexto, indícalo explícitamente."
    )

    # num_ctx: 5 chunks × ~512 tokens + prompt ≈ 3000 tokens de entrada.
    #          16384 da margen para parent_text más largos (PARENT_TEXT_MAX_CHARS
    #          subió a 16000 en nb02) sin reservar VRAM excesiva.
    # num_predict: respuestas RAG cortas; 512 es suficiente.
    GEN_NUM_CTX     = 16384
    GEN_NUM_PREDICT = 512

    gen_llm = ChatOllama(
        model=LLM_GENERATOR_MODEL,
        base_url=OLLAMA_URL,
        reasoning=False,
        num_ctx=GEN_NUM_CTX,
        num_predict=GEN_NUM_PREDICT,
    )

    def call_llm(question: str, contexts: list[str]) -> str:
        context_text = "\n\n---\n\n".join(contexts)
        prompt = (
            f"{SYSTEM_PROMPT}\n\n"
            f"CONTEXTO:\n{context_text}\n\n"
            f"PREGUNTA: {question}\n\nRESPUESTA:"
        )
        return gen_llm.invoke(prompt).content.strip()

    print(f"Generador   : {LLM_GENERATOR_MODEL}")
    print(f"num_ctx     : {GEN_NUM_CTX}  |  num_predict: {GEN_NUM_PREDICT}")
    print(f"reasoning=False  (via ChatOllama)")

In [18]:
if EVAL_MODE not in ("generation", "full"):
    print("[SKIP] EVAL_MODE='retrieval' — loop de generación desactivado.")
else:
    for row in tqdm(retrieval_rows, desc="Generando respuestas"):
        row["response"] = call_llm(row["user_input"], row["retrieved_contexts"])

    print(f"\nRespuestas generadas: {len(retrieval_rows)}")
    print(f"Ejemplo: {retrieval_rows[0]['response'][:300]}...")


Generando respuestas:   0%|          | 0/30 [00:00<?, ?it/s]


Respuestas generadas: 30
Ejemplo: **Respuesta:**

Basado en el contexto proporcionado, **no se identifica a ninguna persona o rol específico** con la responsabilidad explícita de ejecutar un "inventario de la huella de carbono".

El texto incluye secciones sobre gestión de residuos y contaminación con indicaciones para hacer inventa...


In [19]:
if EVAL_MODE not in ("generation", "full"):
    print("[SKIP] EVAL_MODE='retrieval' — guardar cache de respuestas omitido.")
else:
    GEN_CACHE_PATH = RESULTS_DIR / f"responses_{EXPERIMENT_ID}.json"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(GEN_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(retrieval_rows, f, ensure_ascii=False, indent=2, default=str)
    print(f"Cache guardado: {GEN_CACHE_PATH}  ({len(retrieval_rows)} filas)")


Cache guardado: ../../rageval/data/08_reporting/responses_baseline_v0.json  (30 filas)


In [20]:
if EVAL_MODE not in ("generation", "full"):
    print("[SKIP] EVAL_MODE='retrieval' — cargar cache de respuestas omitido.")
else:
    GEN_CACHE_PATH = RESULTS_DIR / f"responses_{EXPERIMENT_ID}.json"
    with open(GEN_CACHE_PATH, encoding="utf-8") as f:
        retrieval_rows = json.load(f)
    print(f"Cache cargado: {GEN_CACHE_PATH}  ({len(retrieval_rows)} filas)")
    print(f"Ejemplo respuesta: {retrieval_rows[0]['response'][:200]}...")


Cache cargado: ../../rageval/data/08_reporting/responses_baseline_v0.json  (30 filas)
Ejemplo respuesta: **Respuesta:**

Basado en el contexto proporcionado, **no se identifica a ninguna persona o rol específico** con la responsabilidad explícita de ejecutar un "inventario de la huella de carbono".

El t...


## 10. Generar respuestas

Loop sobre `retrieval_rows`: llama a `call_llm` para cada pregunta y almacena `response` en el dict.


In [ ]:
if EVAL_MODE not in ("generation", "full"):
    print("[SKIP] EVAL_MODE='retrieval' — evaluación de generación desactivada.")
else:
    from ragas.metrics import (
        ResponseRelevancy,
        Faithfulness,
        FactualCorrectness,
        SemanticSimilarity,
        NoiseSensitivity,
    )

    gen_samples = [
        SingleTurnSample(
            user_input=row["user_input"],
            retrieved_contexts=row["retrieved_contexts"],
            reference_contexts=row["reference_contexts"],
            reference=row["reference"],
            response=row["response"],
        )
        for row in retrieval_rows
    ]
    gen_dataset = EvaluationDataset(samples=gen_samples)

    generation_metrics = [
        ResponseRelevancy(llm=judge_llm, embeddings=ragas_embeddings),
        Faithfulness(llm=judge_llm),
        FactualCorrectness(llm=judge_llm),
        SemanticSimilarity(embeddings=ragas_embeddings),
    ]
    robustness_metrics = [
        NoiseSensitivity(llm=judge_llm),
    ]
    all_gen_metrics = generation_metrics + robustness_metrics

    print(f"Métricas generation : {[m.name for m in generation_metrics]}")
    print(f"Métricas robustez   : {[m.name for m in robustness_metrics]}")
    print(f"Muestras            : {len(gen_dataset)}")
    print()

    gen_result = evaluate(
        dataset=gen_dataset,
        metrics=all_gen_metrics,
        run_config=run_cfg,
    )

    # RAGAS añade sufijos a los nombres (ej: factual_correctness(mode=f1))
    # Derivamos los nombres reales del DataFrame en vez de usar m.name
    _non_metric = {'user_input', 'retrieved_contexts', 'reference_contexts', 'reference', 'response'}
    gen_metric_names = [c for c in gen_result.to_pandas().columns if c not in _non_metric]

    print("\n=== RESULTADOS GENERATION ===")
    print(gen_result)
    print(f"Columnas métricas: {gen_metric_names}")


## 11. Resultados generación


In [ ]:
if EVAL_MODE not in ("generation", "full"):
    print("[SKIP] EVAL_MODE='retrieval' — tablas de generación desactivadas.")
else:
    gen_scores_df = gen_result.to_pandas()
    meta_gen_df   = pd.DataFrame(retrieval_rows)[METADATA_COLS]
    gen_scores_df = pd.concat([gen_scores_df, meta_gen_df.reset_index(drop=True)], axis=1)

    print("── Scores por pregunta ──────────────────────────────────")
    display(gen_scores_df[["user_input"] + gen_metric_names + ["category", "doc_id"]].round(4))

    print()
    print("── Agregados ────────────────────────────────────────────")
    display(gen_scores_df[gen_metric_names].agg(["mean", "std", "min", "max"]).round(4))

    print()
    print("── Por categoría ────────────────────────────────────────")
    display(gen_scores_df.groupby("category")[gen_metric_names].mean().round(4))

    print()
    print("── Por query_type ───────────────────────────────────────")
    display(gen_scores_df.groupby("query_type")[gen_metric_names].mean().round(4))


## 12. Guardar resultados


In [ ]:
if EVAL_MODE not in ("generation", "full"):
    print("[SKIP] EVAL_MODE='retrieval' — guardar resultados generación omitido.")
else:
    _ts = timestamp if 'timestamp' in dir() else datetime.now().strftime("%Y%m%d_%H%M")

    # ── Guardar generación ────────────────────────────────────────────────────
    gen_out_path = RESULTS_DIR / f"results_{EXPERIMENT_ID}_gen_{_ts}.csv"
    gen_scores_df.to_csv(gen_out_path, index=False)
    print(f"Guardado generación : {gen_out_path}")

    # ── Fusionar retrieval + generación ──────────────────────────────────────
    try:
        retr_df = scores_df.copy()
    except NameError:
        candidates = sorted(RESULTS_DIR.glob(f"results_{EXPERIMENT_ID}_[0-9]*.csv"))
        retr_df = pd.read_csv(candidates[-1]) if candidates else None

    if retr_df is not None:
        new_cols = [c for c in gen_scores_df.columns
                    if c not in retr_df.columns and c != "user_input"]
        combined_df = pd.merge(
            retr_df,
            gen_scores_df[["user_input"] + new_cols],
            on="user_input",
            how="inner",
        )
        combined_out = RESULTS_DIR / f"results_{EXPERIMENT_ID}_full_{_ts}.csv"
        combined_df.to_csv(combined_out, index=False)
        print(f"Guardado combinado  : {combined_out}")

    # ── Actualizar resumen acumulado ──────────────────────────────────────────
    summary_path = RESULTS_DIR / "scores_summary.csv"
    if summary_path.exists():
        summary_df = pd.read_csv(summary_path)
        idx = summary_df.index[summary_df["experiment_id"] == EXPERIMENT_ID].tolist()
        row_idx = idx[0] if idx else len(summary_df) - 1
        for m in gen_metric_names:
            summary_df.loc[row_idx, m] = round(gen_scores_df[m].mean(), 4)
    else:
        summary_df = pd.DataFrame([{"experiment_id": EXPERIMENT_ID,
                                     **{m: round(gen_scores_df[m].mean(), 4)
                                        for m in gen_metric_names}}])
    summary_df.to_csv(summary_path, index=False)

    print(f"\nResumen acumulado   : {summary_path}")
    display(summary_df)
